# Final Pipeline Evidence Notebook

This notebook is the evidence notebook for the final diabetic-retinopathy grading pipeline. It does not retrain models. Instead, it loads the committed reports and figures from the reproducible script-based pipeline and summarizes the final model choice.

Final selected run: **class-weighted EfficientNetB3** on APTOS 2019, selected by held-out test QWK.

Important limitation: EfficientNetB3 improves QWK and proliferative recall, but severe-grade recall is lower than the EfficientNetB0 baseline.

## Reproducible Pipeline Sources

The authoritative implementation is script/config based, not notebook based:

- `capstone-project/src/preprocessing.py`
- `capstone-project/src/pre_proc_pipeline.py`
- `capstone-project/src/model.py`
- `capstone-project/src/evaluate.py`
- `capstone-project/src/gradcam.py`
- `capstone-project/configs/efficientnet_b0.json`
- `capstone-project/configs/efficientnet_b1.json`
- `capstone-project/configs/efficientnet_b3.json`
- `capstone-project/configs/efficientnet_b0_smote.json`
- `capstone-project/configs/efficientnet_b0_regression.json`

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "Notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif (PROJECT_DIR / "capstone-project").exists():
    PROJECT_DIR = PROJECT_DIR / "capstone-project"

ARTIFACTS = PROJECT_DIR / "artifacts"
print(PROJECT_DIR)

## Dataset Split Contract

The final pipeline uses APTOS 2019 only. The split metadata below is loaded from the committed preprocessing metadata.

In [ ]:
metadata_path = PROJECT_DIR / "data" / "preprocessed" / "splits" / "metadata.json"
metadata = json.loads(metadata_path.read_text())

split_rows = []
for split_name, split in metadata["splits"].items():
    row = {"split": split_name, "samples": split["saved_samples"]}
    row.update({f"grade_{k}": v for k, v in split["saved_class_distribution"].items()})
    split_rows.append(row)

pd.DataFrame(split_rows)

## Final Results Across Ablations

The table below is loaded directly from the committed `test_report.json` files. QWK is the primary metric.

In [ ]:
runs = {
    "B0 baseline": ARTIFACTS / "test_report.json",
    "SMOTE": ARTIFACTS / "smote" / "test_report.json",
    "Ordinal regression": ARTIFACTS / "regression" / "test_report.json",
    "B1 backbone": ARTIFACTS / "b1" / "test_report.json",
    "B3 backbone": ARTIFACTS / "b3" / "test_report.json",
}

reports = {name: json.loads(path.read_text()) for name, path in runs.items()}
summary = pd.DataFrame([
    {
        "run": name,
        "qwk": report["qwk"],
        "accuracy": report["accuracy"],
        "macro_sensitivity": report["macro_sensitivity"],
        "macro_specificity": report["macro_specificity"],
        "macro_auc": report["macro_auc"],
    }
    for name, report in reports.items()
]).sort_values("qwk", ascending=False)

summary.style.format({
    "qwk": "{:.4f}",
    "accuracy": "{:.4f}",
    "macro_sensitivity": "{:.4f}",
    "macro_specificity": "{:.4f}",
    "macro_auc": "{:.4f}",
})

In [ ]:
per_class_rows = []
for name, report in reports.items():
    row = {"run": name}
    for grade, values in report["per_class"].items():
        row[f"grade_{grade}_sensitivity"] = values["sensitivity"]
    per_class_rows.append(row)

pd.DataFrame(per_class_rows).set_index("run").style.format("{:.4f}")

## Selected Model Evidence: EfficientNetB3

EfficientNetB3 has the highest held-out QWK. The main caveat is severe-grade sensitivity, which drops compared with B0. This should be discussed as a failure mode in the paper.

In [ ]:
b0 = reports["B0 baseline"]
b3 = reports["B3 backbone"]

comparison = pd.DataFrame([
    {"metric": "QWK", "B0": b0["qwk"], "B3": b3["qwk"], "delta": b3["qwk"] - b0["qwk"]},
    {"metric": "Accuracy", "B0": b0["accuracy"], "B3": b3["accuracy"], "delta": b3["accuracy"] - b0["accuracy"]},
    {"metric": "Macro AUC", "B0": b0["macro_auc"], "B3": b3["macro_auc"], "delta": b3["macro_auc"] - b0["macro_auc"]},
    {"metric": "Grade 3 sensitivity", "B0": b0["per_class"]["3"]["sensitivity"], "B3": b3["per_class"]["3"]["sensitivity"], "delta": b3["per_class"]["3"]["sensitivity"] - b0["per_class"]["3"]["sensitivity"]},
    {"metric": "Grade 4 sensitivity", "B0": b0["per_class"]["4"]["sensitivity"], "B3": b3["per_class"]["4"]["sensitivity"], "delta": b3["per_class"]["4"]["sensitivity"] - b0["per_class"]["4"]["sensitivity"]},
])

comparison.style.format({"B0": "{:.4f}", "B3": "{:.4f}", "delta": "{:+.4f}"})

## B3 Evaluation Figures

In [ ]:
for figure in [
    ARTIFACTS / "b3" / "figures" / "confusion_matrix.png",
    ARTIFACTS / "b3" / "figures" / "per_class_metrics.png",
    ARTIFACTS / "b3" / "figures" / "training_curves.png",
]:
    print(figure.name)
    display(Image(filename=str(figure)))

## B3 Explainability Figures

These are qualitative audit artifacts. They must not be interpreted as lesion-mask validation. Grad-CAM and Grad-CAM++ show recurring corner/border activation, which is reported as a failure mode.

In [ ]:
for figure in [
    ARTIFACTS / "b3" / "figures" / "gradcam_summary.png",
    ARTIFACTS / "b3" / "figures" / "gradcampp_summary.png",
    ARTIFACTS / "b3" / "figures" / "scorecam_summary.png",
]:
    print(figure.name)
    display(Image(filename=str(figure)))

## Reproduce Commands

Run from repository root:

```bash
uv run python capstone-project/main.py --out-dir capstone-project/data/preprocessed
uv run python capstone-project/scripts/validate_preprocessed.py

uv run python capstone-project/src/train.py --config capstone-project/configs/efficientnet_b3.json

uv run python capstone-project/src/plots.py \
  --report capstone-project/artifacts/b3/test_report.json \
  --out-dir capstone-project/artifacts/b3/figures \
  --run-name efficientnet-b3-baseline

uv run python capstone-project/src/gradcam.py \
  --model capstone-project/artifacts/b3/best_model.keras \
  --data-dir capstone-project/data/preprocessed \
  --out-dir capstone-project/artifacts/b3/figures \
  --samples-per-grade 6
```
